# WP2b: Sentinel-1 Pre-processing (SAR / Hornsund Track)

**Owner: Julian**

Speckle filtering and GLCM texture feature computation following Williams & Swirad (2025).
Produces a **10-band composite per image** that feeds the SVM classifier in `03b`.

| Band | Description |
|---|---|
| `HH`, `HV` | Raw backscatter (dB) after speckle filter |
| `HH_var`, `HH_contrast`, `HH_ent`, `HH_asm` | GLCM texture on HH |
| `HV_var`, `HV_contrast`, `HV_ent`, `HV_asm` | GLCM texture on HV |

In [ ]:
import ee
import geemap
import sys
import matplotlib.pyplot as plt
import numpy as np

sys.path.insert(0, '..')
from src.utils import load_aoi, get_gee_project
from src.preprocessing_s1 import filter_dual_pol, preprocess_s1, COMPOSITE_BANDS, GLCM_FEATURES

ee.Initialize(project=get_gee_project())
aoi = load_aoi()

## 2b.1 Load S1 dual-pol collection

Filters to IW mode with both HH and HV. `filter_dual_pol()` removes the rare single-pol
images that pass GEE's metadata filter but lack an actual HV band.

In [ ]:
START, END = '2019-01-01', '2024-12-31'

s1_raw = (
    ee.ImageCollection('COPERNICUS/S1_GRD')
    .filterBounds(aoi)
    .filterDate(START, END)
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HH'))
    .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HV'))
    .filter(ee.Filter.eq('instrumentMode', 'IW'))
    .select(['HH', 'HV'])
)
s1_raw = filter_dual_pol(s1_raw)
print('S1 dual-pol images:', s1_raw.size().getInfo())

## 2b.2 Speckle filter + GLCM features

Each image is processed through:
1. **Focal-mean speckle filter** — 50 m radius circle kernel
2. **GLCM texture** — computed on integer-scaled backscatter (9×9 pixel window)
   using the four features from Williams & Swirad (2025): variance, contrast, entropy, ASM

GLCM scaling: backscatter (dB) is shifted by +50 then multiplied by 10, mapping
the typical range [−50, 0] dB to non-negative integers [0, 500].

In [ ]:
s1 = s1_raw.map(preprocess_s1)

print('Bands per image:', s1.first().bandNames().getInfo())
print('Total composite bands:', len(COMPOSITE_BANDS))

## 2b.3 Pick representative winter and summer scenes

We use one winter (January–March) and one summer (July–September) scene throughout
this notebook. These two seasons drive the training data split in `03b`.

In [ ]:
winter = s1.filterDate('2021-01-01', '2021-03-31').first().clip(aoi)
summer = s1.filterDate('2021-07-01', '2021-09-30').first().clip(aoi)

print('Winter scene:', winter.date().format('YYYY-MM-dd').getInfo())
print('Summer scene:', summer.date().format('YYYY-MM-dd').getInfo())

## 2b.4 Visualise backscatter (HH and HV)

In [ ]:
Map = geemap.Map()
Map.centerObject(aoi, zoom=9)

vis_hh = {'min': -25, 'max': 0,  'palette': ['black', 'white']}
vis_hv = {'min': -35, 'max': -5, 'palette': ['black', 'white']}

Map.addLayer(winter.select('HH'), vis_hh, 'HH — Winter')
Map.addLayer(winter.select('HV'), vis_hv, 'HV — Winter')
Map.addLayer(summer.select('HH'), vis_hh, 'HH — Summer', shown=False)
Map.addLayer(summer.select('HV'), vis_hv, 'HV — Summer', shown=False)
Map

## 2b.5 Visualise GLCM texture features

Entropy separates smooth (open water) from rough/heterogeneous (ice) surfaces.
Variance highlights backscatter variability within each ice type.
Compare winter (ice-covered) vs summer (mostly open water) to validate the features.

In [ ]:
Map2 = geemap.Map()
Map2.centerObject(aoi, zoom=9)

Map2.addLayer(winter.select('HH_ent'),      {'min': 0, 'max': 4,    'palette': ['#313695', '#74add1', '#fee090', '#d73027']}, 'HH Entropy — Winter')
Map2.addLayer(winter.select('HH_var'),      {'min': 0, 'max': 3000, 'palette': ['black', 'white']}, 'HH Variance — Winter', shown=False)
Map2.addLayer(winter.select('HH_contrast'), {'min': 0, 'max': 5000, 'palette': ['black', 'white']}, 'HH Contrast — Winter', shown=False)
Map2.addLayer(winter.select('HV_ent'),      {'min': 0, 'max': 4,    'palette': ['#313695', '#74add1', '#fee090', '#d73027']}, 'HV Entropy — Winter', shown=False)
Map2.addLayer(summer.select('HH_ent'),      {'min': 0, 'max': 4,    'palette': ['#313695', '#74add1', '#fee090', '#d73027']}, 'HH Entropy — Summer', shown=False)
Map2

## 2b.6 Band statistics — winter vs summer

A quick sanity check: GLCM features should differ noticeably between seasons
if they carry useful information for ice classification.

In [ ]:
def get_band_means(image, bands, scale=200):
    stats = image.select(bands).reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=aoi,
        scale=scale,
        maxPixels=1e10,
        bestEffort=True,
    )
    return stats.getInfo()

w_means = get_band_means(winter, COMPOSITE_BANDS)
s_means = get_band_means(summer, COMPOSITE_BANDS)

print(f'{'Band':<20} {'Winter':>10} {'Summer':>10}')
print('-' * 42)
for band in COMPOSITE_BANDS:
    wv = w_means.get(band)
    sv = s_means.get(band)
    wstr = f'{wv:.3f}' if wv is not None else 'N/A'
    sstr = f'{sv:.3f}' if sv is not None else 'N/A'
    print(f'{band:<20} {wstr:>10} {sstr:>10}')

In [ ]:
glcm_bands = [b for b in COMPOSITE_BANDS if b not in ('HH', 'HV')]

w_vals = [w_means.get(b, 0) or 0 for b in glcm_bands]
s_vals = [s_means.get(b, 0) or 0 for b in glcm_bands]

x = np.arange(len(glcm_bands))
fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(x - 0.2, w_vals, 0.4, label='Winter', color='#4575b4')
ax.bar(x + 0.2, s_vals, 0.4, label='Summer', color='#d73027')
ax.set_xticks(x)
ax.set_xticklabels(glcm_bands, rotation=45, ha='right')
ax.set_ylabel('Mean value over AOI')
ax.set_title('GLCM feature means — winter vs summer')
ax.legend()
plt.tight_layout()
plt.show()

## 2b.7 (Optional) Export preprocessed collection to GEE asset

Processing the GLCM on the fly in `03b` is fine for single-image inspection,
but slow if you call `.getInfo()` over many images. Export the collection to a
GEE ImageCollection asset first to speed up the classification workflow.

> Only needed if you experience timeouts in `03b`. Leave commented for now.

In [ ]:
# Exports each image as a separate GEE asset. Run once, then load by asset path in 03b.
#
# def export_image(image):
#     date_str = image.date().format('YYYYMMdd').getInfo()
#     task = ee.batch.Export.image.toAsset(
#         image=image.select(COMPOSITE_BANDS).clip(aoi).toFloat(),
#         description=f's1_glcm_{date_str}',
#         assetId=f'projects/<your-project>/assets/s1_glcm/{date_str}',
#         region=aoi,
#         scale=50,
#         crs='EPSG:4326',
#         maxPixels=1e13,
#     )
#     task.start()
#     return task
#
# images = s1.toList(s1.size())
# n = s1.size().getInfo()
# for i in range(n):
#     export_image(ee.Image(images.get(i)))
# print(f'Started {n} export tasks.')
print('Export: uncomment and fill in your asset path when needed.')